# Bonus 03 — Serve Your Own Model with vLLM

**Optional | After Lab 4 and Lab 5 | Colab T4 GPU | No API key**

> **Switch to a T4 before you run anything:** Runtime → Change runtime type → T4 GPU → Save.

---

Lab 5 Part C described vLLM and never ran it: Colab's CPU runtime cannot. This notebook runs it, on the free T4, serving the same Qwen model you fine-tuned in Lab 4, and with your adapter merged in if you still have it.

Then you do the three things Lab 5 could only talk about:

1. Call your own GPU-hosted model with the OpenAI client, the `base_url` swap one more time.
2. Read how vLLM divides the GPU's memory between weights and KV cache, and how many users that buys you.
3. Fire 32 requests at once and measure what continuous batching actually does to throughput.

**Coming from Lab 4:** you saved an adapter to `./my_lora_adapter`, and Part D of that lab said some runtimes want a plain merged model instead of base + adapter. vLLM on a T4 is one of them (section 2 explains why), so that is the artifact you will build here.

**Coming from Lab 5:** the server below is started the same way as yours: a background process, a log file, and a health check before anything calls it.

> This notebook was written against the vLLM docs of September 2026 and checked on CPU wherever possible, but it has **not yet been run end to end on a Colab T4**. If a cell fails, the troubleshooting table at the end covers the likely causes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

**Checkpoint:** you should see `Tesla T4` and about 15,000 MiB. If you see an error, the runtime has no GPU. Change it now: changing it later wipes everything you have installed.

The install is big. vLLM brings its own build of PyTorch and CUDA kernels, so expect **three to five minutes**. `peft` is for merging your adapter in section 2.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} vllm peft openai httpx

---

## 1. Choose what to serve

`BASE_MODEL` must be the model your Lab 4 adapter was trained on: `Qwen/Qwen2.5-1.5B-Instruct`, unless you switched Lab 4 to 0.5B.

`ADAPTER_SOURCE` says where your adapter comes from:

| Value | When to use it |
|---|---|
| `"upload"` | You still have Lab 4's runtime or its files. In **Lab 4's** notebook, run `!zip -r my_lora_adapter.zip my_lora_adapter` and download the zip from the Files panel. You will upload it in the next cell. No account needed. |
| `"your-name/my-qlora-adapter"` | You pushed it to the Hugging Face Hub (Lab 4 stretch goal 4). A free account is enough for a public repo. |
| `None` | No adapter. Everything below still works with the base model; section 2 is skipped. |

In [ ]:
BASE_MODEL     = "Qwen/Qwen2.5-1.5B-Instruct"   # must match what Lab 4 trained on
ADAPTER_SOURCE = None                           # "upload", a Hub repo id, or None

In [ ]:
import zipfile

ADAPTER_DIR = None
if ADAPTER_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()                           # pick my_lora_adapter.zip
    zipfile.ZipFile(next(iter(uploaded))).extractall(".")
    ADAPTER_DIR = "./my_lora_adapter"
elif ADAPTER_SOURCE:
    from huggingface_hub import snapshot_download
    ADAPTER_DIR = snapshot_download(ADAPTER_SOURCE)

print("Adapter:", ADAPTER_DIR or "none, serving the base model")

---

## 2. Merge the adapter into the model

vLLM *can* serve adapters directly: start it with `--enable-lora --lora-modules lab4=./my_lora_adapter`, and clients pick the adapter by passing `model="lab4"`. One base model in GPU memory, many adapters on top, which is how multi-tenant fine-tune hosting works. It is the natural next step after Lab 4.

On a T4 it is unreliable. vLLM's LoRA kernels are compiled with Triton, and on the T4's older architecture that compilation has failed for exactly this setup: a Qwen2.5 model with an adapter ([vLLM issue #20259](https://github.com/vllm-project/vllm/issues/20259), closed without a fix). On an L4 or A100 the direct route is the better one.

So we build the artifact Lab 4 Part D called the **merged model**: add `B·A` into each weight once, and save an ordinary model folder. vLLM then serves it like any other model. The trade-off, from Lab 4: one merged copy per fine-tune instead of one small adapter each.

The merge runs on the **CPU** on purpose. Loading anything onto the GPU here would hold GPU memory in this notebook that the vLLM server, a separate process, needs.

In [ ]:
SERVE_MODEL = BASE_MODEL

if ADAPTER_DIR:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map="cpu")
    merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
    merged.save_pretrained("./merged_model")
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained("./merged_model")
    del base, merged
    SERVE_MODEL = "./merged_model"

print("Serving:", SERVE_MODEL)

**Checkpoint:** with an adapter, `Serving: ./merged_model`. That folder is about 3 GB for the 1.5B model, the same number Lab 4 printed for its merged export.

One honest caveat. Your adapter was trained against a **4-bit** base, and here it is merged into a **16-bit** one. The answers will be close to Lab 4's, not identical. Teams that care about exact parity merge into the same precision they serve.

---

## 3. Start the server

The same pattern as Lab 5: start it in the background, send its output to a log file, and wait for `/health`. Each flag is something an earlier lab measured:

| Flag | Why | Where you met it |
|---|---|---|
| `--dtype half` | The T4 has no native bf16; vLLM refuses to start in bf16 on it | Lab 4 (compute dtype) |
| `--max-model-len 2048` | Longest conversation to plan for; caps KV cache per request | Lab 3 (KV cache) |
| `--gpu-memory-utilization 0.85` | vLLM reserves this share of GPU memory up front, then carves the KV cache out of what the weights leave | Lab 5 Part C (PagedAttention) |
| `--served-model-name qwen-lab4` | The name clients put in `model=` | Lab 1A (`model` is just a string) |

Startup takes **one to three minutes**: download the weights (or read the merged folder), load them, profile memory, and warm up the GPU kernels.

In [ ]:
import os, shutil, subprocess, time, httpx

# the vllm command-line tool the install put on PATH (on Colab: /usr/local/bin/vllm)
VLLM = shutil.which("vllm") or os.path.join(os.path.dirname(sys.executable), "vllm")
cmd = [VLLM, "serve", SERVE_MODEL,
       "--dtype", "half",
       "--max-model-len", "2048",
       "--gpu-memory-utilization", "0.85",
       "--served-model-name", "qwen-lab4",
       "--port", "8000"]

log = open("vllm.log", "w")
server_proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

for _ in range(120):                                   # up to 10 minutes
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            print("vLLM is up")
            break
    except httpx.HTTPError:
        pass
    if server_proc.poll() is not None:
        print("vLLM exited. Last lines of vllm.log:")
        print("".join(open("vllm.log").readlines()[-25:]))
        break
    time.sleep(5)

**Checkpoint:** `vLLM is up`. If it exited, the log lines tell you why; see the troubleshooting table at the end.

Before calling it, read what vLLM decided about memory. It logs how many tokens of KV cache fit after the weights, and how many full-length conversations that allows at once.

In [ ]:
!grep -iE "kv cache|concurrency|loading took" vllm.log | tail -6

**Checkpoint:** look for how much memory the weights took, a KV cache size in tokens, and a line like `Maximum concurrency for 2,048 tokens per request`. That second number is the Lab 3 calculation, done by the server: GPU memory left after the weights, divided by the KV cache one conversation needs. Write it down; section 5 will push against it.

---

## 4. Same client, one more `base_url`

The OpenAI client from Lab 1A, pointed at your GPU. vLLM does not check the key, so any string works.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="not-needed")
print([m.id for m in client.models.list().data])

In [ ]:
def ask(question, max_tokens=150):
    r = client.chat.completions.create(model="qwen-lab4", max_tokens=max_tokens, temperature=0,
                                       messages=[{"role": "user", "content": question}])
    return r.choices[0].message.content

for q in ["What is QLoRA?", "How does LoRA reduce trainable parameters?", "What is NF4 quantization?"]:
    print("Q:", q)
    print("A:", ask(q), "\n")

**Checkpoint:** these are Lab 4's three test questions. If you merged your adapter, compare the answers with what Lab 4's `TUNED` column printed: they should have the same style, with small differences from the 4-bit vs 16-bit merge. Without an adapter, this is the base model, and it will be confidently vague, as Lab 3 warned.

Everything from here on is the course's one idea again. Lab 5's FastAPI proxy could sit in front of this server with `BACKEND_BASE_URL=http://127.0.0.1:8000/v1`. Bonus 04's LiteLLM could route to it as `hosted_vllm/qwen-lab4`. The client code would not change.

---

## 5. Continuous batching, measured

Lab 5 claimed that a naive server makes users queue for the GPU, and that vLLM does not. Here is the test.

Two runs with the same prompt and the same answer length:

- **One at a time:** 8 requests, each waiting for the previous one to finish. This is what a `generate()` call inside a FastAPI handler gives you.
- **All at once:** 32 requests sent together. vLLM batches them step by step: every decoding step advances all active requests.

Before you run it: if 8 requests take *T* seconds one after another, how long will 32 take all at once?

In [ ]:
import asyncio
from openai import AsyncOpenAI

aclient = AsyncOpenAI(base_url="http://127.0.0.1:8000/v1", api_key="not-needed")
PROMPT = [{"role": "user", "content": "Explain what a KV cache is, in about 100 words."}]

async def one_request():
    r = await aclient.chat.completions.create(model="qwen-lab4", messages=PROMPT,
                                              max_tokens=128, temperature=0.7)
    return r.usage.completion_tokens

async def run(n, all_at_once):
    t0 = time.perf_counter()
    if all_at_once:
        tokens = await asyncio.gather(*[one_request() for _ in range(n)])
    else:
        tokens = [await one_request() for _ in range(n)]
    seconds = time.perf_counter() - t0
    return sum(tokens), seconds

In [ ]:
tok_seq, sec_seq = await run(8, all_at_once=False)
print(f"one at a time : {8:>2} requests, {sec_seq:5.1f} s, {tok_seq / sec_seq:6.0f} tokens/s")

tok_par, sec_par = await run(32, all_at_once=True)
print(f"all at once   : {32:>2} requests, {sec_par:5.1f} s, {tok_par / sec_par:6.0f} tokens/s")

print(f"\nthroughput x{(tok_par / sec_par) / (tok_seq / sec_seq):.1f} with 4x the requests")

**Checkpoint:** four times the requests, in about the same time as the eight sequential ones or less, and a throughput several times higher. Write down your multiple.

Why: generating one token for one request barely uses the T4. Most of each step is spent reading the weights from GPU memory. Reading them once and applying them to 32 requests costs about the same as applying them to one. Continuous batching keeps that batch full, adding new requests as others finish, and PagedAttention makes room for all their KV caches. That is Lab 5 Part C, measured on hardware you control.

It does not go on forever. Push past the maximum concurrency from section 3 and requests start waiting for KV cache space. Try 128 in the stretch goals.

---

## 6. What the server reports about itself

vLLM publishes live metrics at `/metrics` in the Prometheus format that monitoring systems scrape. This is Lab 8's observability idea from the server's side: how many requests are running and waiting, and how full the KV cache is.

In [ ]:
metrics = httpx.get("http://127.0.0.1:8000/metrics").text
for line in metrics.splitlines():
    if line.startswith("vllm:") and any(k in line for k in ("num_requests_running", "num_requests_waiting", "cache_usage", "request_success_total")):
        print(line)

**Checkpoint:** the `request_success_total` lines (split by why each request stopped) should add up to 43: the 3 questions in section 4 plus the 8 and 32 of the benchmark. Running and waiting are back at 0 now that the benchmark is over. In production these are the numbers you alert on: waiting requests climbing means you need another GPU.

---

## 7. Stop the server

The server holds almost all of the T4's memory until it stops.

In [ ]:
server_proc.terminate()
server_proc.wait(timeout=60)
print("vLLM stopped")

---

## Troubleshooting

| What you see in `vllm.log` | Likely cause | Fix |
|---|---|---|
| `Bfloat16 is only supported on GPUs with compute capability of at least 8.0` | `--dtype half` missing | Keep the flag; the T4 cannot do bf16 |
| `CUDA out of memory` at startup | Too much reserved, or something else on the GPU | Restart the runtime (this notebook should hold no GPU memory itself), or lower `--gpu-memory-utilization` to 0.7 or `--max-model-len` to 1024 |
| Health check times out, log still busy | Slow first download or kernel warm-up | Wait; or add `--enforce-eager`, which skips CUDA graph capture: faster start, somewhat slower serving |
| `No CUDA GPUs are available` | The runtime has no GPU | Change the runtime type to T4 and start again |
| Merge fails with a size mismatch | `BASE_MODEL` differs from what Lab 4 trained on | Check `base_model_name_or_path` in the adapter's `adapter_config.json` |

## Bonus 03 complete

- [ ] vLLM served your model on a T4 and answered through the OpenAI client
- [ ] You read the KV cache size and maximum concurrency from the log
- [ ] You measured the throughput gain from sending requests together
- [ ] You can say why this notebook merged the adapter instead of serving it directly

## Stretch

1. **Find the ceiling.** Run the all-at-once benchmark with 64 and 128 requests. Where does the gain level off, and how does that compare with the maximum concurrency from section 3?
2. **Shorter context, more users.** Restart with `--max-model-len 1024`. What happens to the maximum concurrency in the log?
3. **Put Lab 5 in front.** Start Lab 5's `server.py` in this runtime on port 8001 with `BACKEND_BASE_URL=http://127.0.0.1:8000/v1`, `BACKEND_API_KEY=anything` and `DEFAULT_MODEL=qwen-lab4`, then call *that*. Your own API, your own model, your own GPU.
4. **On a bigger GPU**, if you have access to an L4 or A100: serve the adapter directly with `--enable-lora --lora-modules lab4=./my_lora_adapter --max-lora-rank 16` and call `model="lab4"`.

Next: [Bonus 04 — LiteLLM Gateway](04_litellm_gateway.ipynb) puts one gateway in front of OpenAI, Groq and a server like this one.